In [2]:
from bs4 import BeautifulSoup
import pandas as pd
import requests as rq
import json

In [4]:
url = 'https://boardgamegeek.com/boardgame/176443'
test = rq.get(url)
test.content
soup = BeautifulSoup(test.content, 'html.parser')
element = soup.find_all('link')
element[0].get('href')

'https://boardgamegeek.com/boardgame/176443/descent-journeys-in-the-dark-second-edition-dark-e'

STEP 1: Go to this URL: https://boardgamegeek.com/boardgamemechanic/2023/cooperative-game/linkeditems/boardgamemechanic?pageid=1&sort=rank
STEP 2: Open up the developer view using F12
STEP 3: Click on the Network tab
STEP 4: Filter traffic to 'Fetch/XHR'
STEP 5: Get all the objectids into a list to call apis for top 100 games
STEP 6: Repeat the process for the bottom 100 games
STEP 7: Take the objectid and get the name of the game to form the url for the comments with the form: https://boardgamegeek.com/boardgame/{objectid}/{name-with-dashes-as-spaces}/ratings?pageid=1&comment=1&rated=1&status=own
STEP 8: Store data in a database for later analysis


In [3]:
top_objects = [
    161936, 174430, 291457, 162886, 167355, 295770, 205637, 192135, 96848, 324856, 285774,
    314040, 255984, 221107, 253344, 205059, 251661, 373106, 209010, 284083, 55690, 191189,
    121921, 264220, 146021
]

In [4]:
bottom_objects = [
    176443, 3225, 160589, 163021, 215370, 135523, 162604, 104413, 43533, 388470, 219575,
    249201, 186151, 59967, 177163, 321697, 365139, 162225, 357291, 339094, 313301, 158969,
    339130, 295284, 156416
]

In [6]:
top_name_id_urls = []
for object in top_objects:
    url = f'https://boardgamegeek.com/boardgame/{object}'
    req = rq.get(url)
    soup = BeautifulSoup(req.content, 'html.parser')
    link = soup.find('link')
    href = link.get('href')
    top_name_id_urls.append(str(href))

In [7]:
bottom_name_id_urls = []
for object in bottom_objects:
    url = f'https://boardgamegeek.com/boardgame/{object}'
    req = rq.get(url)
    soup = BeautifulSoup(req.content, 'html.parser')
    link = soup.find('link')
    href = link.get('href')
    bottom_name_id_urls.append(str(href))

Looking like I am going to have to use selenium or something similar bc the comments are javascript rendered material

In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [13]:
columns = ['GAME_ID', 'GAME_NAME', 'COMMENT', 'RATING']
main_df = pd.DataFrame(columns=columns)

In [9]:
driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()))


Confirmed the above method work for opening up the webpage. Now I need to work on automating the page navigation and getting all the comments. Probably looking at setting up a data base for this at this point. Some have upwards of 55k comments.

In [4]:
comments = driver.find_elements(By.CSS_SELECTOR, 'li:nth-child(4) div:nth-child(3) div:nth-child(1) div:nth-child(1) div:nth-child(1) p:nth-child(1)')

So the key to getting the comments is going to be using this line "li:nth-child(<list element number>)". I need to iterate over all the comments in the list (which I can probably also get this with selenium -> try using this css selector to get the total number of comments: span[class='ng-isolate-scope'] strong[class='ng-binding']). This should give me all the comments.

I'll also need to handle what happens when we need to navigate pages but we'll save that for another time.

In [3]:
num_comments = driver.find_element(By.CSS_SELECTOR, "span[class='ng-isolate-scope'] strong[class='ng-binding']")
num_comments.text

'3,902'

In [20]:
comments[0].text

'My favorite co-op game!'

In [19]:
import time

def get_num_comments(game: str) -> int:
    
    full_url = game + '/ratings?pageid=1&comment=1&rated=1&status=own'
    driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()))
    driver.get(full_url)
    time.sleep(5)
    num_comments = driver.find_element(By.CSS_SELECTOR, "span[class='ng-isolate-scope'] strong[class='ng-binding']")

    return int((num_comments.text).replace(',', ''))

for game in top_name_id_urls:

    print(get_num_comments(game))


3902
6081
2639
4477
2455
825
3668
1149
3199
1178
1874
445
955
1031
1067
2456
731
687
1306
2790
1237
1500
3470
1286
2947


In [14]:
import uuid
test = uuid.uuid4()

print(test)

41398e38-b461-4347-aeba-dbb58ddddd89


In [22]:
import time
import uuid
import pickle as pk

row = []
for game in top_name_id_urls:
    
    game_url = game
    num_comments = get_num_comments(game_url)
    driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()))
    
    for page in range(1, num_comments // 50 + 2):
        
        driver.get(game_url + f'/ratings?pageid={page}&comment=1&rated=1&status=own')
        
        looking_for_name = True
        while(looking_for_name):
            try:
                game_name = (driver.find_element(By.CSS_SELECTOR, "h1 span[class='ng-binding']")).text
                looking_for_name = False
            except:
                pass

        looking_for_rating = True
        while(looking_for_rating):
            try:
                game_rating = (driver.find_element(By.CSS_SELECTOR, ".ng-binding[ng-attr-itemprop=\"{{item.stats.usersrated > 0 ? 'ratingValue' : undefined}}\"]")).text
                looking_for_rating = False
            except:
                pass
            
        id = uuid.uuid4()
        
        for comment in range(1, 51):
            
            try:
                review = (driver.find_element(By.CSS_SELECTOR, f'li:nth-child({comment}) div:nth-child(3) div:nth-child(1) div:nth-child(1) div:nth-child(1) p:nth-child(1)')).text
            except:
                pass
            row.extend([id, game_name, review, game_rating])
            main_df.loc[-1] = row
            main_df.index = main_df.index + 1
            row.clear()

        time.sleep(3)

with open('reviews.pkl','wb') as file:
    pk.dump(main_df, file)

The approach above worked. Just need to loop through all the games in my list. Expect this to take a significant amount of time. Took 17 minutes to get all the comments for pandemic legacy.

In [3]:
data['CMT_LEN'] = data.apply(lambda x: len(x['COMMENT']), axis=1)

In [5]:
data['KEY'] = data.apply(lambda x: str(x['GAME_ID']) + x['COMMENT'] + str(x['CMT_LEN']), axis=1)

In [11]:
data_undup = data.drop_duplicates()

In [12]:
data_undup

,GAME_ID,GAME_NAME,COMMENT,RATING,CMT_LEN,KEY
94599,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,37.1 x 27.0 x 7.6 cm,8.5,20,6c485b5a-283e-4f21-98cf-b058efcfe14737.1 x 27....
94598,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,I was favorably surprised by this. I didn't bu...,8.5,792,6c485b5a-283e-4f21-98cf-b058efcfe147I was favo...
94597,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite in the Pandemic Legacy series and ...,8.5,318,6c485b5a-283e-4f21-98cf-b058efcfe147My favorit...
94596,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite co-op game!,8.5,23,6c485b5a-283e-4f21-98cf-b058efcfe147My favorit...
94595,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,This is a must buy legacy game for anyone that...,8.5,131,6c485b5a-283e-4f21-98cf-b058efcfe147This is a ...
...,...,...,...,...,...,...
53,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Fantastic game. If you are looking for the evo...,7.8,354,3d5214d6-ec58-4d25-9b79-602e2a2d6387Fantastic ...
52,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Arkham Horror sprinkled with a choose-your-own...,7.8,128,3d5214d6-ec58-4d25-9b79-602e2a2d6387Arkham Hor...
51,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,"Not really a boardgame in fact, EH should be s...",7.8,210,3d5214d6-ec58-4d25-9b79-602e2a2d6387Not really...
50,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Watered down version of Arkham Horror. Still m...,7.8,168,3d5214d6-ec58-4d25-9b79-602e2a2d6387Watered do...


In [13]:
data_undup = data_undup.reset_index()
data_undup.sort_index(inplace=True, ascending=True)

In [16]:
data_undup = data_undup.drop(columns='index')
data_undup

,GAME_ID,GAME_NAME,COMMENT,RATING,CMT_LEN,KEY
0,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,37.1 x 27.0 x 7.6 cm,8.5,20,6c485b5a-283e-4f21-98cf-b058efcfe14737.1 x 27....
1,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,I was favorably surprised by this. I didn't bu...,8.5,792,6c485b5a-283e-4f21-98cf-b058efcfe147I was favo...
2,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite in the Pandemic Legacy series and ...,8.5,318,6c485b5a-283e-4f21-98cf-b058efcfe147My favorit...
3,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite co-op game!,8.5,23,6c485b5a-283e-4f21-98cf-b058efcfe147My favorit...
4,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,This is a must buy legacy game for anyone that...,8.5,131,6c485b5a-283e-4f21-98cf-b058efcfe147This is a ...
...,...,...,...,...,...,...
76343,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Fantastic game. If you are looking for the evo...,7.8,354,3d5214d6-ec58-4d25-9b79-602e2a2d6387Fantastic ...
76344,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Arkham Horror sprinkled with a choose-your-own...,7.8,128,3d5214d6-ec58-4d25-9b79-602e2a2d6387Arkham Hor...
76345,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,"Not really a boardgame in fact, EH should be s...",7.8,210,3d5214d6-ec58-4d25-9b79-602e2a2d6387Not really...
76346,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Watered down version of Arkham Horror. Still m...,7.8,168,3d5214d6-ec58-4d25-9b79-602e2a2d6387Watered do...


In [18]:
import string

translator = str.maketrans('','', string.punctuation)
data_undup['COMMENT'] = data_undup.apply(lambda x: x['COMMENT'].translate(translator),axis=1)
data_undup

,GAME_ID,GAME_NAME,COMMENT,RATING,CMT_LEN,KEY
0,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,371 x 270 x 76 cm,8.5,20,6c485b5a-283e-4f21-98cf-b058efcfe14737.1 x 27....
1,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,I was favorably surprised by this I didnt buy ...,8.5,792,6c485b5a-283e-4f21-98cf-b058efcfe147I was favo...
2,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite in the Pandemic Legacy series and ...,8.5,318,6c485b5a-283e-4f21-98cf-b058efcfe147My favorit...
3,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite coop game,8.5,23,6c485b5a-283e-4f21-98cf-b058efcfe147My favorit...
4,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,This is a must buy legacy game for anyone that...,8.5,131,6c485b5a-283e-4f21-98cf-b058efcfe147This is a ...
...,...,...,...,...,...,...
76343,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Fantastic game If you are looking for the evol...,7.8,354,3d5214d6-ec58-4d25-9b79-602e2a2d6387Fantastic ...
76344,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Arkham Horror sprinkled with a chooseyourownad...,7.8,128,3d5214d6-ec58-4d25-9b79-602e2a2d6387Arkham Hor...
76345,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Not really a boardgame in fact EH should be se...,7.8,210,3d5214d6-ec58-4d25-9b79-602e2a2d6387Not really...
76346,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Watered down version of Arkham Horror Still ma...,7.8,168,3d5214d6-ec58-4d25-9b79-602e2a2d6387Watered do...


In [19]:
data_undup = data_undup.drop(columns = 'KEY')
data_undup

,GAME_ID,GAME_NAME,COMMENT,RATING,CMT_LEN
0,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,371 x 270 x 76 cm,8.5,20
1,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,I was favorably surprised by this I didnt buy ...,8.5,792
2,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite in the Pandemic Legacy series and ...,8.5,318
3,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite coop game,8.5,23
4,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,This is a must buy legacy game for anyone that...,8.5,131
...,...,...,...,...,...
76343,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Fantastic game If you are looking for the evol...,7.8,354
76344,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Arkham Horror sprinkled with a chooseyourownad...,7.8,128
76345,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Not really a boardgame in fact EH should be se...,7.8,210
76346,3d5214d6-ec58-4d25-9b79-602e2a2d6387,Eldritch Horror,Watered down version of Arkham Horror Still ma...,7.8,168


In [20]:
with open('reviews.pkl','wb') as file:
    pk.dump(data_undup, file)